# Project Title
### Explore Data

#### Project Summary
--describe your project at a high level--

The project follows the follow steps:
* Step 1: Scope the Project and Gather Data
* Step 2: Explore and Assess the Data
* Step 3: Define the Data Model
* Step 4: Run ETL to Model the Data
* Step 5: Complete Project Write Up

In [1]:
import time

In [2]:
# Do all imports and installs here
import pandas as pd
import os
from pyspark.sql.types import DateType,StringType
from pyspark.sql.functions import isnan, count, when, col,lit,udf,array
from pyspark.sql.functions import year, month, dayofmonth, hour, weekofyear, dayofweek, date_format, monotonically_increasing_id, to_date

### Step 1: Scope the Project and Gather Data

#### Scope 
Explain what you plan to do in the project in more detail. What data do you use? What is your end solution look like? What tools did you use? etc>

#### Describe and Gather Data 
Describe the data sets you're using. Where did it come from? What type of information is included? 

1. **I94 Immigration Data** : This data comes from the US National Tourism and Trade Office. A data dictionary is included in the workspace. This is where the data comes from. There's a sample file so you can take a look at the data in csv format before reading it all in. You do not have to use the entire dataset, just use what you need to accomplish the goal you set at the beginning of the project.
2. **World Temperature Data** : This dataset came from Kaggle. You can read more about it here.
3. **U.S. City Demographic Data** : This data comes from OpenSoft. You can read more about it here.
4. **Airport Code Table** : This is a simple table of airport codes and corresponding cities. It comes from here.


In [4]:
#Immigration Data

# For this project, we will only read i94_apr16_sub.sas7bdat
# If all files need to be processed then df_f can be used

fname = '../../data/18-83510-I94-Data-2016/i94_apr16_sub.sas7bdat'

df_spark = spark.read.format('com.github.saurfang.sas.spark').load(fname)

#all files
# be careful with running this cell, will take a long time to execute
#read all the files in the dir
"""
files = os.listdir('../../data/18-83510-I94-Data-2016')
for name in files:
    # read the data into a data frame
    fname = '../../data/18-83510-I94-Data-2016/' + name
    df_f =  spark.read.format('com.github.saurfang.sas.spark').load(fname)
"""

In [ ]:
#World Temperature data
fname = '../../data2/GlobalLandTemperaturesByCity.csv'

In [4]:
# Demographics data here
fname = "us-cities-demographics.csv"


In [ ]:
#Airport Data
fname="airport-codes_csv.csv"

In [7]:
#below command to used to write data in parquet format in sas_data dir and then read from it
#write to parquet
#df_spark.write.parquet("sas_data")
df_spark=spark.read.parquet("sas_data")


In [5]:
df_spark.head()

Row(cicid=6.0, i94yr=2016.0, i94mon=4.0, i94cit=692.0, i94res=692.0, i94port='XXX', arrdate=20573.0, i94mode=None, i94addr=None, depdate=None, i94bir=37.0, i94visa=2.0, count=1.0, dtadfile=None, visapost=None, occup=None, entdepa='T', entdepd=None, entdepu='U', matflag=None, biryear=1979.0, dtaddto='10282016', gender=None, insnum=None, airline=None, admnum=1897628485.0, fltno=None, visatype='B2')

In [9]:
df_spark.count()

3096313

In [6]:
df_spark.createOrReplaceTempView("i94_table")

In [8]:
#spark.sql("select count(1) from i94_table").show()

#lot of invalid state columns or null columns. We will replace  state values that are not in the dim_state table with code 99
spark.sql("select i94addr,count(1) from i94_table group by i94addr order by i94addr").show(500)

+-------+--------+
|i94addr|count(1)|
+-------+--------+
|   null|  152592|
|     ..|      30|
|     .C|      17|
|     .D|       2|
|     .I|       4|
|     .L|       2|
|     .M|       2|
|     .N|       4|
|     .T|       6|
|      0|      26|
|     00|       1|
|     02|       1|
|     03|       1|
|     06|       1|
|     10|      12|
|     11|       4|
|     14|       1|
|     18|       1|
|     20|       5|
|     21|       4|
|     23|       2|
|     27|       1|
|      3|       2|
|     30|       3|
|     32|       1|
|     33|       3|
|     36|       1|
|     3H|       1|
|     40|       1|
|     4B|       2|
|     52|       1|
|     53|       1|
|     60|       1|
|     61|       1|
|     63|       1|
|     70|       1|
|     71|       1|
|     72|       1|
|     73|       1|
|     77|       1|
|     81|       1|
|     85|       1|
|     89|       1|
|     90|       2|
|     91|       1|
|     93|       1|
|     97|       1|
|     99|      52|
|      A|       1|
|     AA|   

In [49]:
df_spark.printSchema()

root
 |-- cicid: double (nullable = true)
 |-- i94yr: double (nullable = true)
 |-- i94mon: double (nullable = true)
 |-- i94cit: double (nullable = true)
 |-- i94res: double (nullable = true)
 |-- i94port: string (nullable = true)
 |-- arrdate: double (nullable = true)
 |-- i94mode: double (nullable = true)
 |-- i94addr: string (nullable = true)
 |-- depdate: double (nullable = true)
 |-- i94bir: double (nullable = true)
 |-- i94visa: double (nullable = true)
 |-- count: double (nullable = true)
 |-- dtadfile: string (nullable = true)
 |-- visapost: string (nullable = true)
 |-- occup: string (nullable = true)
 |-- entdepa: string (nullable = true)
 |-- entdepd: string (nullable = true)
 |-- entdepu: string (nullable = true)
 |-- matflag: string (nullable = true)
 |-- biryear: double (nullable = true)
 |-- dtaddto: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- insnum: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- admnum: double (nullable = 

### Step 2: Explore and Assess the Data
#### Explore the Data 
Identify data quality issues, like missing values, duplicate data, etc.

#### Cleaning Steps
Document steps necessary to clean the data

In [2]:
#Create SparkSession

from pyspark.sql import SparkSession

spark = SparkSession.builder.\
config("spark.jars.repositories", "https://repos.spark-packages.org/").\
config("spark.jars.packages", "saurfang:spark-sas7bdat:2.0.0-s_2.11").\
enableHiveSupport().getOrCreate()



## Immigration Data Analysis

In [16]:
# list all files in the customer repository
files = os.listdir('../../data/18-83510-I94-Data-2016')
files

['i94_apr16_sub.sas7bdat',
 'i94_sep16_sub.sas7bdat',
 'i94_nov16_sub.sas7bdat',
 'i94_mar16_sub.sas7bdat',
 'i94_jun16_sub.sas7bdat',
 'i94_aug16_sub.sas7bdat',
 'i94_may16_sub.sas7bdat',
 'i94_jan16_sub.sas7bdat',
 'i94_oct16_sub.sas7bdat',
 'i94_jul16_sub.sas7bdat',
 'i94_feb16_sub.sas7bdat',
 'i94_dec16_sub.sas7bdat']

In [6]:
# Performing cleaning tasks here

fname = '../../data/18-83510-I94-Data-2016/i94_apr16_sub.sas7bdat'

## to import the file in pandas
df = pd.read_sas(fname, 'sas7bdat', encoding="ISO-8859-1")

## to import the file in spark
df_spark = spark.read.format('com.github.saurfang.sas.spark').load(fname)

In [ ]:
#Parse SAS label description file
#Read I94_SAS_Labels_Descriptions.SAS



#### Read SAS description file to create dim_country, dim_state and dim_port

In [9]:
#Read I94_SAS_Labels_Descriptions.SAS
with open("I94_SAS_Labels_Descriptions.SAS") as f:
    desc = f.readlines()
    
#desc holds the contents of the entire file

#dim_country data is in lines 10-298
#dim_port is in lines 303-962
#dim_state is in lines 982-1036

In [11]:
country_dict = {}
for countries in desc[10:298]:
    data=countries.split("=")
    country_code,country_name = data[0].strip(),data[1].strip().strip("'")
    country_dict[country_code]=country_name
    
country_df = pd.DataFrame(list(country_dict.items()),columns=['country_code','country_name'])


In [12]:
country_df.head()

,country_code,country_name
0,236,AFGHANISTAN
1,101,ALBANIA
2,316,ALGERIA
3,102,ANDORRA
4,324,ANGOLA


In [14]:
port_dict = {}
for ports in desc[303:962]:
    data=ports.split("=")
    port_code,port_name = data[0].strip().strip("'"),data[1].strip("\t").strip().strip("'").strip()
    port_dict[port_code]=port_name
    
port_df = pd.DataFrame(list(port_dict.items()),columns=['port_code','port_name'])

In [15]:
port_df.head()

,port_code,port_name
0,ANC,"ANCHORAGE, AK"
1,BAR,"BAKER AAF - BAKER ISLAND, AK"
2,DAC,"DALTONS CACHE, AK"
3,PIZ,"DEW STATION PT LAY DEW, AK"
4,DTH,"DUTCH HARBOR, AK"


In [17]:
state_dict = {}
for states in desc[982:1036]:
    data=states.split("=")
    state_code,state_name = data[0].strip("\t").strip("'"),data[1].strip().strip("'")
    state_dict[state_code]=state_name
    
state_df = pd.DataFrame(list(state_dict.items()),columns=['state_code','state_name'])

In [78]:
state_df_spark = spark.createDataFrame(state_df)
state_df_spark.head()

Row(state_code='AK')

In [14]:
df.shape

(3096313, 28)

In [15]:
df.head()

,cicid,i94yr,i94mon,i94cit,i94res,i94port,arrdate,i94mode,i94addr,depdate,...,entdepu,matflag,biryear,dtaddto,gender,insnum,airline,admnum,fltno,visatype
0,6.0,2016.0,4.0,692.0,692.0,XXX,20573.0,NaN,NaN,NaN,...,U,NaN,1979.0,10282016,NaN,NaN,NaN,1.897628e+09,NaN,B2
1,7.0,2016.0,4.0,254.0,276.0,ATL,20551.0,1.0,AL,NaN,...,Y,NaN,1991.0,D/S,M,NaN,NaN,3.736796e+09,00296,F1
2,15.0,2016.0,4.0,101.0,101.0,WAS,20545.0,1.0,MI,20691.0,...,NaN,M,1961.0,09302016,M,NaN,OS,6.666432e+08,93,B2
3,16.0,2016.0,4.0,101.0,101.0,NYC,20545.0,1.0,MA,20567.0,...,NaN,M,1988.0,09302016,NaN,NaN,AA,9.246846e+10,00199,B2
4,17.0,2016.0,4.0,101.0,101.0,NYC,20545.0,1.0,MA,20567.0,...,NaN,M,2012.0,09302016,NaN,NaN,AA,9.246846e+10,00199,B2


In [ ]:
# be careful with running this cell, will take a long time to execute
#read and get count of records for all the files in the dir
for name in files:
    # read the data into a data frame
    fname = '../../data/18-83510-I94-Data-2016/' + name
    df_f =  pd.read_sas(fname, 'sas7bdat', encoding="ISO-8859-1")
    print(f'{fname} df.shape = ', df_f.shape)

../../data/18-83510-I94-Data-2016/i94_apr16_sub.sas7bdat df.shape =  (3096313, 28)


In [20]:
df_spark.count()

3096313

In [7]:
# Set the threshold for missing values (90%)
threshold = 0.9

# Calculate the percentage of missing values for each column
missing_percentage = df.isnull().mean()

# Filter columns that have more than 90% missing values
columns_with_high_missing = missing_percentage[missing_percentage > threshold].index.tolist()

print("Columns with more than 90% missing data:", columns_with_high_missing)



Columns with more than 90% missing data: ['occup', 'entdepu', 'insnum']


In [8]:
#drop these columns
#df_spark.count()
#new_immig_df = df_spark.drop(*columns_with_high_missing)
df_spark = df_spark.drop(*columns_with_high_missing)

#new_immig_df.count()

In [42]:
df_spark.printSchema()

root
 |-- cicid: double (nullable = true)
 |-- i94yr: double (nullable = true)
 |-- i94mon: double (nullable = true)
 |-- i94cit: double (nullable = true)
 |-- i94res: double (nullable = true)
 |-- i94port: string (nullable = true)
 |-- arrdate: double (nullable = true)
 |-- i94mode: double (nullable = true)
 |-- i94addr: string (nullable = true)
 |-- depdate: double (nullable = true)
 |-- i94bir: double (nullable = true)
 |-- i94visa: double (nullable = true)
 |-- count: double (nullable = true)
 |-- dtadfile: string (nullable = true)
 |-- visapost: string (nullable = true)
 |-- entdepa: string (nullable = true)
 |-- entdepd: string (nullable = true)
 |-- matflag: string (nullable = true)
 |-- biryear: double (nullable = true)
 |-- dtaddto: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- admnum: double (nullable = true)
 |-- fltno: string (nullable = true)
 |-- visatype: string (nullable = true)



In [11]:
#Drop duplicates SPARK DF
df_spark = df_spark.dropDuplicates(['cicid'])

2.4318695068359375e-05


In [24]:
start_time = time.time()
#alternate (slower) way to check using sql
print(spark.sql("select cicid,count(1) from i94_table group by cicid having count(1) >1").show())

end_time = time.time()
#execution time
print(end_time - start_time)

+-----+--------+
|cicid|count(1)|
+-----+--------+
+-----+--------+

None
22.027432203292847


In [38]:
#IN PANDAS drop rows with missing cicid values
#get count first
print(df['cicid'].isnull().sum())

#drop
df = df.dropna(how='all', subset=['cicid'])

0


In [26]:
#IN SPARK DF drop rows with missing cicid values
#get count first
null_count = df_spark.filter(col('cicid').isNull()).count()
print(null_count)

#drop
df_spark = df_spark.na.drop(subset=['cicid'])

0


In [50]:
#get update df count
print(df.shape)
print(df_spark.count())

(3096313, 28)
3096313


In [ ]:
#clean i94_addr column with valid states based on states in state_df_spark

In [96]:
#create udf that can be used in automating the pipeline
@udf(StringType())
def validate_state(x,valid_states):
    if x in valid_states:
        return x
    return "99"


In [82]:
valid_states = state_df["state_code"].unique()

In [85]:
print(valid_states)

['AK' 'AZ' 'AR' 'CA' 'CO' 'CT' 'DE' 'DC' 'FL' 'GA' 'GU' 'HI' 'ID' 'IL' 'IN'
 'IA' 'KS' 'KY' 'LA' 'ME' 'MD' 'MA' 'MI' 'MN' 'MS' 'MO' 'MT' 'NC' 'ND' 'NE'
 'NV' 'NH' 'NJ' 'NM' 'NY' 'OH' 'OK' 'OR' 'PA' 'PR' 'RI' 'SC' 'SD' 'TN' 'TX'
 'UT' 'VT' 'VI' 'VA' 'WV' 'WA' 'WI' 'WY' '99']


In [89]:
#clean up state column "i94addr" in fact_immi
clean_df_spark = df_spark.withColumn("i94addr",validate_state(df_spark.i94addr))

In [26]:
clean_df_spark.count()

3096313

In [90]:
clean_df_spark.limit(5).toPandas()

,cicid,i94yr,i94mon,i94cit,i94res,i94port,arrdate,i94mode,i94addr,depdate,...,entdepa,entdepd,matflag,biryear,dtaddto,gender,airline,admnum,fltno,visatype
0,6.0,2016.0,4.0,692.0,692.0,XXX,20573.0,NaN,99,NaN,...,T,None,None,1979.0,10282016,None,None,1.897628e+09,None,B2
1,7.0,2016.0,4.0,254.0,276.0,ATL,20551.0,1.0,99,NaN,...,G,None,None,1991.0,D/S,M,None,3.736796e+09,00296,F1
2,15.0,2016.0,4.0,101.0,101.0,WAS,20545.0,1.0,MI,20691.0,...,T,O,M,1961.0,09302016,M,OS,6.666432e+08,93,B2
3,16.0,2016.0,4.0,101.0,101.0,NYC,20545.0,1.0,MA,20567.0,...,O,O,M,1988.0,09302016,None,AA,9.246846e+10,00199,B2
4,17.0,2016.0,4.0,101.0,101.0,NYC,20545.0,1.0,MA,20567.0,...,O,O,M,2012.0,09302016,None,AA,9.246846e+10,00199,B2


In [59]:
#df_main = df_main.withColumn('is_inref', when(col('main').isin(mask_vl_list), 'YES').otherwise('NO'))

#df_spark = df_spark.withColumn('i94addr_new', when(~(col('i94addr').isin(state_code_list)),'99') \
 #                                 .otherwise(col('i94addr')))

In [44]:
#spark.sql("select i94addr,count(1) from i94_table group by i94addr order by i94addr").show(57)
#spark.sql("select count(1) from i94_table where i94addr='99' ").show()

+--------+
|count(1)|
+--------+
|      52|
+--------+



In [43]:
#convert the unknow i94addr state values to 99 by joining to state_df
#first add an id column for i94addr to join to state_df_spark df
df_spark.withColumn("id",col("i94addr"))

Row(cicid=6.0, i94yr=2016.0, i94mon=4.0, i94cit=692.0, i94res=692.0, i94port='XXX', arrdate=20573.0, i94mode=None, i94addr=None, depdate=None, i94bir=37.0, i94visa=2.0, count=1.0, dtadfile=None, visapost=None, entdepa='T', entdepd=None, matflag=None, biryear=1979.0, dtaddto='10282016', gender=None, airline=None, admnum=1897628485.0, fltno=None, visatype='B2', id=None)

In [ ]:
df_spark.head(5)

In [92]:
clean_df_spark.createOrReplaceTempView("i94_table_convertedstate")

In [93]:
#spark.sql("select i94addr,count(1) from i94_table group by i94addr order by i94addr").show(500)
spark.sql("select i94addr,count(1) from i94_table_convertedstate group by i94addr order by i94addr").show(500)

+-------+--------+
|i94addr|count(1)|
+-------+--------+
|     99|  187354|
|     AK|    1604|
|     AR|    2873|
|     AZ|   20218|
|     CA|  470386|
|     CO|   15874|
|     CT|   13991|
|     DC|   28228|
|     DE|    3111|
|     FL|  621701|
|     GA|   44663|
|     GU|   94107|
|     HI|  168764|
|     IA|    3391|
|     ID|    1752|
|     IL|   82126|
|     IN|   11278|
|     KS|    3224|
|     KY|    5790|
|     LA|   22655|
|     MA|   70486|
|     MD|   25360|
|     ME|    2361|
|     MI|   32101|
|     MN|   11194|
|     MO|    8484|
|     MS|    1771|
|     MT|    1339|
|     NC|   23375|
|     ND|    1225|
|     NE|   26574|
|     NH|    2817|
|     NJ|   76531|
|     NM|    1994|
|     NV|  114609|
|     NY|  553677|
|     OH|   18089|
|     OK|    3239|
|     OR|   12574|
|     PA|   30293|
|     PR|    9474|
|     RI|    3289|
|     SC|    9811|
|     SD|     557|
|     TN|   12105|
|     TX|  134321|
|     UT|    7551|
|     VA|   31399|
|     VI|     226|
|     VT|   

## World Temperature Data Analysis

In [4]:
fname = '../../data2/GlobalLandTemperaturesByCity.csv'
spark_df = spark.read.csv(fname,header=True,inferSchema=True)
panda_df = pd.read_csv(fname)


In [5]:
print(spark_df.count()) #spark df
print(panda_df.shape) #panda df

8599212
(8599212, 7)


In [7]:
spark_df.printSchema()

root
 |-- dt: timestamp (nullable = true)
 |-- AverageTemperature: double (nullable = true)
 |-- AverageTemperatureUncertainty: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)



In [8]:
spark_df.show(n=5)

+-------------------+------------------+-----------------------------+-----+-------+--------+---------+
|                 dt|AverageTemperature|AverageTemperatureUncertainty| City|Country|Latitude|Longitude|
+-------------------+------------------+-----------------------------+-----+-------+--------+---------+
|1743-11-01 00:00:00|             6.068|           1.7369999999999999|Århus|Denmark|  57.05N|   10.33E|
|1743-12-01 00:00:00|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
|1744-01-01 00:00:00|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
|1744-02-01 00:00:00|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
|1744-03-01 00:00:00|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
+-------------------+------------------+-----------------------------+-----+-------+--------+---------+
only showing top 5 rows



In [6]:
panda_df.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
0,1743-11-01,6.068,1.737,Århus,Denmark,57.05N,10.33E
1,1743-12-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
2,1744-01-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
3,1744-02-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
4,1744-03-01,NaN,NaN,Århus,Denmark,57.05N,10.33E


In [10]:
# Set the threshold for missing values (90%)
threshold = 0.9

# Calculate the percentage of missing values for each column
#concert spark df to panda df to run the  mean command
#converting spark to panda brings the whote dataset in memory so its not good for huge datasets
#temparature dataset is too large to better to use pandas df to get the column names
#missing_percentage = spark_df.toPandas().isnull().mean()


#using panda df
missing_percentage = panda_df.isnull().mean()


# Filter columns that have more than 90% missing values
columns_with_high_missing = missing_percentage[missing_percentage > threshold].index.tolist()

print("Columns with more than 90% missing data:", columns_with_high_missing)

#drop these columns
#panda_df = panda_df.drop(*columns_with_high_missing)

Columns with more than 90% missing data: []


In [11]:
#check for row count where average temperature data is missing and drop it
spark_df.filter(col("AverageTemperature").isNull()).count()

364130

In [5]:
spark_df = spark_df.dropna(subset=['AverageTemperature'])
spark_df.count()

8235082

In [6]:
#drop records where there is duplicate data for dt,city,country
spark_df = spark_df.drop_duplicates(subset=['dt','City','Country'])
spark_df.count()

8190783

In [7]:
spark_df.limit(5).toPandas()

,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
0,1743-11-01,5.354,1.636,Frankfurt,Germany,50.63N,8.87E
1,1743-11-01,7.030,1.611,Münster,Germany,52.24N,7.88E
2,1743-11-01,5.044,2.222,Pitesti,Romania,44.20N,24.60E
3,1743-11-01,13.808,1.918,Sabadell,Spain,40.99N,2.13E
4,1744-04-01,10.352,2.222,Akron,United States,40.99N,80.95W


In [8]:
##Exclude data prior to 1960-1-1. (assumption is we don't have travel records prior to that)
temp_spark_df = spark_df.filter(to_date(col('dt')) >= to_date(lit('1960-01-01')) )

In [13]:
temp_spark_df.head()

Row(dt=datetime.datetime(1960, 1, 1, 0, 0), AverageTemperature=24.592, AverageTemperatureUncertainty=0.317, City='Benguela', Country='Angola', Latitude='12.05S', Longitude='13.15E')

In [14]:
temp_spark_df.limit(5).toPandas()

,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
0,1960-01-01,24.592,0.317,Benguela,Angola,12.05S,13.15E
1,1960-01-01,19.149,0.590,Bhusawal,India,21.70N,75.29E
2,1960-01-01,23.405,0.541,Blumenau,Brazil,26.52S,48.36W
3,1960-01-01,27.865,0.362,Bohicon,Benin,7.23N,2.43E
4,1960-01-01,18.735,0.431,Chhindwara,India,21.70N,78.75E


In [9]:
 temp_spark_df.createOrReplaceTempView("temp_spark_df_view")

In [12]:
#Create dim_temperature dimension table

dim_temperature = temp_spark_df.select('dt','AverageTemperature', \
                                       'AverageTemperatureUncertainty', \
                                       'City','Country' ) \
                                .withColumnRenamed('dt','date') \
                                .withColumnRenamed('AverageTemperature','avg_temp') \
                                .withColumnRenamed('AverageTemperatureUncertainty','avg_temp_uncertainty') \
                                .withColumnRenamed('City','city') \
                                .withColumnRenamed('Country','country')

dim_temperature = dim_temperature.withColumn('date',to_date(col('date')))
dim_temperature = dim_temperature.withColumn('year',year(dim_temperature['date']))
dim_temperature = dim_temperature.withColumn('month',month(dim_temperature['date']))

+-------------------+------------------+-----------------------------+---------+-----------+--------+---------+
|                 dt|AverageTemperature|AverageTemperatureUncertainty|     City|    Country|Latitude|Longitude|
+-------------------+------------------+-----------------------------+---------+-----------+--------+---------+
|1960-02-01 00:00:00|            26.279|                        0.508|  Angeles|Philippines|  15.27N|  120.83E|
|1960-02-01 00:00:00|             2.033|                        0.153|Bielefeld|    Germany|  52.24N|    7.88E|
+-------------------+------------------+-----------------------------+---------+-----------+--------+---------+



In [ ]:
dim_temperature.head()

## US City Demographics Analysis

In [8]:
fname = "us-cities-demographics.csv"

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.\
config("spark.jars.repositories", "https://repos.spark-packages.org/").\
config("spark.jars.packages", "saurfang:spark-sas7bdat:2.0.0-s_2.11").\
enableHiveSupport().getOrCreate()

In [24]:
spark_df = spark.read.csv(fname,header=True,inferSchema=True,sep=';')
panda_df = pd.read_csv(fname,sep=';')

In [25]:
print(spark_df.count()) #spark df
print(panda_df.shape) #panda df

2891
(2891, 12)


In [26]:
spark_df.printSchema()

root
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Median Age: double (nullable = true)
 |-- Male Population: integer (nullable = true)
 |-- Female Population: integer (nullable = true)
 |-- Total Population: integer (nullable = true)
 |-- Number of Veterans: integer (nullable = true)
 |-- Foreign-born: integer (nullable = true)
 |-- Average Household Size: double (nullable = true)
 |-- State Code: string (nullable = true)
 |-- Race: string (nullable = true)
 |-- Count: integer (nullable = true)



In [30]:
spark_df.limit(5).toPandas()

,City,State,Median Age,Male Population,Female Population,Total Population,Number of Veterans,Foreign-born,Average Household Size,State Code,Race,Count
0,Silver Spring,Maryland,33.8,40601,41862,82463,1562,30908,2.60,MD,Hispanic or Latino,25924
1,Quincy,Massachusetts,41.0,44129,49500,93629,4147,32935,2.39,MA,White,58723
2,Hoover,Alabama,38.5,38040,46799,84839,4819,8229,2.58,AL,Asian,4759
3,Rancho Cucamonga,California,34.5,88127,87105,175232,5821,33878,3.18,CA,Black or African-American,24437
4,Newark,New Jersey,34.6,138040,143873,281913,5829,86253,2.73,NJ,White,76402


In [38]:
spark_df.toPandas().isnull().mean()
#five columns have a few nulls so we can delete those rows
"""
Male Population           0.001038
Female Population         0.001038
Number of Veterans        0.004497
Foreign-born              0.004497
Average Household Size    0.005534
"""

##alternate way to find columns counts
spark_df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in spark_df.columns]
   ).toPandas()

City                      0.000000
State                     0.000000
Median Age                0.000000
Male Population           0.001038
Female Population         0.001038
Total Population          0.000000
Number of Veterans        0.004497
Foreign-born              0.004497
Average Household Size    0.005534
State Code                0.000000
Race                      0.000000
Count                     0.000000
dtype: float64

In [39]:
spark_df = spark_df.dropna(subset=[
    'Male Population',
    'Female Population',
    'Number of Veterans',
    'Foreign-born',
    'Average Household Size'
])
spark_df.count()

2875

In [40]:
spark_df = spark_df.drop_duplicates(subset=['City','State','State Code','Race'])
spark_df.count()

2875

In [ ]:
dim_cities_population = spark_df.select('City','State','Male Population', \
                                                 'Female Population', \
                                                 'Number of Veterans','Foreign-born','Race') \
                                         .withColumnRenamed('City','city') \
                                         .withColumnRenamed('State','state') \
                                         .withColumnRenamed('Male Population','male_population') \
                                         .withColumnRenamed('Female Population','female_population') \
                                         .withColumnRenamed('Number of Veterans','num_of_veterans') \
                                         .withColumnRenamed('Foreign-born','foreign_born') \
                                         .withColumnRenamed('Race','race')
    

In [ ]:
dim_cities_population.head()

## Airport Code Table Analysis

In [4]:
fname="airport-codes_csv.csv"

In [43]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.\
config("spark.jars.repositories", "https://repos.spark-packages.org/").\
config("spark.jars.packages", "saurfang:spark-sas7bdat:2.0.0-s_2.11").\
enableHiveSupport().getOrCreate()

In [5]:
spark_df = spark.read.csv(fname,header=True,inferSchema=True,sep=',')
panda_df = pd.read_csv(fname,sep=',')

In [6]:
spark_df.printSchema()

root
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- coordinates: string (nullable = true)



In [7]:
print(spark_df.count()) #spark df
print(panda_df.shape) #panda df

55075
(55075, 12)


In [8]:
spark_df.limit(5).toPandas()

,ident,type,name,elevation_ft,continent,iso_country,iso_region,municipality,gps_code,iata_code,local_code,coordinates
0,00A,heliport,Total Rf Heliport,11,NA,US,US-PA,Bensalem,00A,None,00A,"-74.93360137939453, 40.07080078125"
1,00AA,small_airport,Aero B Ranch Airport,3435,NA,US,US-KS,Leoti,00AA,None,00AA,"-101.473911, 38.704022"
2,00AK,small_airport,Lowell Field,450,NA,US,US-AK,Anchor Point,00AK,None,00AK,"-151.695999146, 59.94919968"
3,00AL,small_airport,Epps Airpark,820,NA,US,US-AL,Harvest,00AL,None,00AL,"-86.77030181884766, 34.86479949951172"
4,00AR,closed,Newport Hospital & Clinic Heliport,237,NA,US,US-AR,Newport,None,None,None,"-91.254898, 35.6087"


In [62]:
#check for duplicate values in ident column
spark_df.groupBy("ident").count().filter(col("count") > 1).count()

#no duplicates found


0

In [67]:
#Drop duplicates
spark_df = spark_df.dropDuplicates(['ident'])
spark_df.count()

55075

In [74]:
#columns with null values
spark_df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in spark_df.columns]
               ).toPandas()

,ident,type,name,elevation_ft,continent,iso_country,iso_region,municipality,gps_code,iata_code,local_code,coordinates
0,0,0,0,7006,0,0,0,5676,14045,45886,26389,0


In [14]:
#countries
spark_df.groupBy("iso_country").count().show()
spark_df.groupBy("iso_country").count().filter(col("iso_country") == 'US').show()

+-----------+-----+
|iso_country|count|
+-----------+-----+
|         DZ|   61|
|         LT|   55|
|         MM|   75|
|         CI|   26|
|         TC|   10|
|         AZ|   35|
|         FI|  111|
|         SC|   18|
|         PM|    2|
|         UA|  191|
|         ZM|  103|
|         KI|   22|
|         RO|   59|
|         SL|   12|
|         SB|   38|
|         NL|   90|
|         LA|   20|
|         BS|   66|
|         BW|  127|
|         MN|   29|
+-----------+-----+
only showing top 20 rows

+-----------+-----+
|iso_country|count|
+-----------+-----+
|         US|22757|
+-----------+-----+



In [15]:
#since our immigration dataset only has US arrival airports we will only use data for US airports
spark_df = spark_df.filter(spark_df['iso_country'] =='US')

In [19]:

spark_df.count()

22757

In [24]:
#check for null municipalities
spark_df.limit(5).toPandas()
spark_df.filter(col("municipality").isNull()).count()


102

In [25]:
spark_df = spark_df.dropna(subset=['municipality'])

In [26]:
spark_df.count()

22655

In [ ]:
dim_airports = spark_df.select('ident','type','name', \
                                           'continent','iso_country','iso_region','municipality') \
                                    .withColumnRenamed('ident','airport_code') \
                                    .withColumnRenamed('type','airport_type') \
                                    .withColumnRenamed('name','airport_name') \
                                    .withColumnRenamed('continent','continent') \
                                    .withColumnRenamed('iso_country','country') \
                                    .withColumnRenamed('iso_region','region') \
                                    .withColumnRenamed('municipality','city')

In [ ]:
dim_airports.head()

### Step 3: Define the Data Model
#### 3.1 Conceptual Data Model
Map out the conceptual data model and explain why you chose that model

#### 3.2 Mapping Out Data Pipelines
List the steps necessary to pipeline the data into the chosen data model

### Step 4: Run Pipelines to Model the Data 
#### 4.1 Create the data model
Build the data pipelines to create the data model.

In [ ]:
# Write code here

#### 4.2 Data Quality Checks
Explain the data quality checks you'll perform to ensure the pipeline ran as expected. These could include:
 * Integrity constraints on the relational database (e.g., unique key, data type, etc.)
 * Unit tests for the scripts to ensure they are doing the right thing
 * Source/Count checks to ensure completeness
 
Run Quality Checks

In [ ]:
# Perform quality checks here

#### 4.3 Data dictionary 
Create a data dictionary for your data model. For each field, provide a brief description of what the data is and where it came from. You can include the data dictionary in the notebook or in a separate file.

#### Step 5: Complete Project Write Up
* Clearly state the rationale for the choice of tools and technologies for the project.
* Propose how often the data should be updated and why.
* Write a description of how you would approach the problem differently under the following scenarios:
 * The data was increased by 100x.
 * The data populates a dashboard that must be updated on a daily basis by 7am every day.
 * The database needed to be accessed by 100+ people.